In [1]:
%load_ext autoreload
%autoreload 2
    
import os
import sys
import pandas as pd
import xarray as xr
import numpy as np
from dask.distributed import Client
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from collections import defaultdict
from pathlib import Path
sys.path.append('/home/563/sc1326/repos/cdrmip_extremes')
from cdrmip_extremes.configs import data_dir, proc_dir
from cdrmip_extremes.sat import find_crossing_years, find_matching_gwls
from cdrmip_extremes import processing, utils, ext_freq, utils_scenariomip
from cdrmip_extremes.CMIP6_file_search import find_CMIP_files, search_and_load, search_and_load_parent, concatenate_branches

In [2]:
client = Client(memory_limit=None,threads_per_worker=1,n_workers=28)

# Load ssp534-over and ssp126 extension tas data

In [3]:
expts = ['ssp534-over','ssp126']
chunks = {'time':365}
vars_to_drop=['height','time_bnds','lat_bnds','lon_bnds','areacella']
table_id = 'Amon'
variable_id = 'tas'

In [4]:
data = {}
for experiment_id in expts:
    data[experiment_id] = search_and_load(
        experiment_id, table_id, variable_id, vars_to_drop, chunks,
        threshold=2102, check_end_year=True
    )

Searching for ssp534-over files
Loading ACCESS-CM2 r1i1p1f1
ACCESS-CM2 r1i1p1f1: 2041-01-16 12:00:00 → 2100-12-16 12:00:00
  Skipping — end year 2100 not beyond 2102
Loading ACCESS-ESM1-5 r10i1p1f2
ACCESS-ESM1-5 r10i1p1f2: 2041-01-16 12:00:00 → 2300-12-16 12:00:00
Loading ACCESS-ESM1-5 r11i1p1f2
ACCESS-ESM1-5 r11i1p1f2: 2041-01-16 12:00:00 → 2300-12-16 12:00:00
Loading ACCESS-ESM1-5 r12i1p1f2
ACCESS-ESM1-5 r12i1p1f2: 2041-01-16 12:00:00 → 2300-12-16 12:00:00
Loading ACCESS-ESM1-5 r13i1p1f2
ACCESS-ESM1-5 r13i1p1f2: 2041-01-16 12:00:00 → 2300-12-16 12:00:00
Loading ACCESS-ESM1-5 r14i1p1f2
ACCESS-ESM1-5 r14i1p1f2: 2041-01-16 12:00:00 → 2300-12-16 12:00:00
Loading ACCESS-ESM1-5 r15i1p1f2
ACCESS-ESM1-5 r15i1p1f2: 2041-01-16 12:00:00 → 2300-12-16 12:00:00
Loading ACCESS-ESM1-5 r16i1p1f2
ACCESS-ESM1-5 r16i1p1f2: 2041-01-16 12:00:00 → 2300-12-16 12:00:00
Loading ACCESS-ESM1-5 r17i1p1f2
ACCESS-ESM1-5 r17i1p1f2: 2041-01-16 12:00:00 → 2300-12-16 12:00:00
Loading ACCESS-ESM1-5 r18i1p1f2
ACCESS-ESM

## Concatenate

In [5]:
# Step 1: chain ssp534-over -> ssp585
for model, member_dict in data['ssp534-over'].items():
    for member, child_ds in member_dict.items():
        parent_ds = search_and_load_parent(child_ds,chunks=chunks,vars_to_drop=vars_to_drop)
        print(f"Concatenating ssp534-over to {parent_ds.attrs['experiment_id']} [{model}/{member}]")
        if model == 'ACCESS-ESM1-5':
            branch_date = '2041-01-01 00:00:00'
        else:
            branch_date=None
        data['ssp534-over'][model][member] = concatenate_branches(
            child_ds, parent_ds, time_dim='time',
            branch_date=branch_date
        )

# Step 2: chain all experiments -> historical
concat = defaultdict(lambda: defaultdict(dict))
for expt, model_dict in data.items():
    for model, member_dict in model_dict.items():
        for member, child_ds in member_dict.items():
            parent_ds = search_and_load_parent(child_ds,chunks=chunks,vars_to_drop=vars_to_drop)
            print(f"Concatenating {expt} to {parent_ds.attrs['experiment_id']} [{model}/{member}]")
            branch_date = '2015-01-01'
            concat[model][expt][member] = concatenate_branches(
                child_ds, parent_ds, time_dim='time',
                branch_date=branch_date,
            )

Seaching for ACCESS-ESM1-5 ssp534-over r10i1p1f2 parent
Loading ACCESS-ESM1-5 ssp585 r10i1p1f1
Concatenating ssp534-over to ssp585 [ACCESS-ESM1-5/r10i1p1f2]
Seaching for ACCESS-ESM1-5 ssp534-over r11i1p1f2 parent
Loading ACCESS-ESM1-5 ssp585 r11i1p1f1
Concatenating ssp534-over to ssp585 [ACCESS-ESM1-5/r11i1p1f2]
Seaching for ACCESS-ESM1-5 ssp534-over r12i1p1f2 parent
Loading ACCESS-ESM1-5 ssp585 r12i1p1f1
Concatenating ssp534-over to ssp585 [ACCESS-ESM1-5/r12i1p1f2]
Seaching for ACCESS-ESM1-5 ssp534-over r13i1p1f2 parent
Loading ACCESS-ESM1-5 ssp585 r13i1p1f1
Concatenating ssp534-over to ssp585 [ACCESS-ESM1-5/r13i1p1f2]
Seaching for ACCESS-ESM1-5 ssp534-over r14i1p1f2 parent
Loading ACCESS-ESM1-5 ssp585 r14i1p1f1
Concatenating ssp534-over to ssp585 [ACCESS-ESM1-5/r14i1p1f2]
Seaching for ACCESS-ESM1-5 ssp534-over r15i1p1f2 parent
Loading ACCESS-ESM1-5 ssp585 r15i1p1f1
Concatenating ssp534-over to ssp585 [ACCESS-ESM1-5/r15i1p1f2]
Seaching for ACCESS-ESM1-5 ssp534-over r16i1p1f2 parent
Lo

In [6]:
concat = utils.defaultdict_to_dict(concat)

## load piControl

In [8]:
# load piControl recognising it is often the same for ssp126, ssp534-over
piControl = defaultdict(dict)

for model, expt_dict in concat.items():
    for expt, member_dict in expt_dict.items():
        for member, child_ds in member_dict.items():
            
            parent_ds = search_and_load_parent(
                child_ds,
                chunks=chunks,
                vars_to_drop=vars_to_drop,
            )

            if parent_ds is None:
                continue

            if member not in piControl[model]:
                piControl[model][member] = parent_ds

            elif not piControl[model][member].identical(parent_ds):
                raise ValueError(
                    f"Inconsistent piControl for {model} {member}"
                )
piControl = utils.defaultdict_to_dict(piControl)


Seaching for ACCESS-ESM1-5 historical r10i1p1f1 parent
Loading ACCESS-ESM1-5 piControl r1i1p1f1
Seaching for ACCESS-ESM1-5 historical r11i1p1f1 parent
Loading ACCESS-ESM1-5 piControl r1i1p1f1
Seaching for ACCESS-ESM1-5 historical r12i1p1f1 parent
Loading ACCESS-ESM1-5 piControl r1i1p1f1
Seaching for ACCESS-ESM1-5 historical r13i1p1f1 parent
Loading ACCESS-ESM1-5 piControl r1i1p1f1
Seaching for ACCESS-ESM1-5 historical r14i1p1f1 parent
Loading ACCESS-ESM1-5 piControl r1i1p1f1
Seaching for ACCESS-ESM1-5 historical r15i1p1f1 parent
Loading ACCESS-ESM1-5 piControl r1i1p1f1
Seaching for ACCESS-ESM1-5 historical r16i1p1f1 parent
Loading ACCESS-ESM1-5 piControl r1i1p1f1
Seaching for ACCESS-ESM1-5 historical r17i1p1f1 parent
Loading ACCESS-ESM1-5 piControl r1i1p1f1
Seaching for ACCESS-ESM1-5 historical r18i1p1f1 parent
Loading ACCESS-ESM1-5 piControl r1i1p1f1
Seaching for ACCESS-ESM1-5 historical r19i1p1f1 parent
Loading ACCESS-ESM1-5 piControl r1i1p1f1
Seaching for ACCESS-ESM1-5 historical r1

## Create DataTree

In [9]:
dt = xr.DataTree.from_dict({
    f"/{model}/{expt}/{member}": ds
    for model, expt_dict in concat.items()
    for expt, member_dict in expt_dict.items()
    for member, ds in member_dict.items()
})

In [10]:
pi_datatree = xr.DataTree.from_dict({
    f"/{model}/{member}": ds
    for model, member_dict in piControl.items()
    for member, ds in member_dict.items()
})

## Regrid to common grid

In [11]:
def regrid_ds(ds):
    if 'tas' in ds:
        return processing.regrid_to_common(ds)
    return ds

regridded_dt = dt.map_over_datasets(regrid_ds)
regridded_pi_dt = pi_datatree.map_over_datasets(regrid_ds)

Building regridder for grid (145,) x (192,)
Building regridder for grid (192,) x (288,)
Building regridder for grid (128,) x (256,)
Building regridder for grid (64,) x (128,)
Building regridder for grid (256,) x (512,)
Building regridder for grid (90,) x (144,)
Building regridder for grid (143,) x (144,)
Building regridder for grid (64,) x (128,)
Building regridder for grid (160,) x (320,)
Building regridder for grid (144,) x (192,)


## Calculate Anomalies

In [12]:
reference_period = slice(1850,1900)
def anomaly_func(ds: xr.Dataset) -> xr.Dataset:
    if 'tas' not in ds:
        return ds
    # Annual means — collapses 'time' dim into 'year'
    ds_annual = ds.groupby('time.year').mean(dim='time')
    return utils.calc_anomaly(ds_annual, ds_annual, reference_period, time_dim='year')

anomaly_dt = regridded_dt.map_over_datasets(anomaly_func)

# Calculate global mean

In [13]:
def global_mean(ds: xr.Dataset) -> xr.Dataset:
    if 'tas' not in ds:
        return ds
    return utils.global_mean(ds).to_dataset(name='gsat')

gsat_dt = anomaly_dt.map_over_datasets(global_mean)

## Save Data

In [19]:
processing.save_processed(anomaly_dt,proc_dir,'ScenarioMIP','anomalies','tas')

Skipping (already exists): /home/563/sc1326/repos/cdrmip_extremes/data/processed/ScenarioMIP/tas/anomalies/ACCESS-ESM1-5/ACCESS-ESM1-5_ssp534-over_r10i1p1f2_tas_anomalies.nc
Skipping (already exists): /home/563/sc1326/repos/cdrmip_extremes/data/processed/ScenarioMIP/tas/anomalies/ACCESS-ESM1-5/ACCESS-ESM1-5_ssp534-over_r11i1p1f2_tas_anomalies.nc
Skipping (already exists): /home/563/sc1326/repos/cdrmip_extremes/data/processed/ScenarioMIP/tas/anomalies/ACCESS-ESM1-5/ACCESS-ESM1-5_ssp534-over_r12i1p1f2_tas_anomalies.nc
Skipping (already exists): /home/563/sc1326/repos/cdrmip_extremes/data/processed/ScenarioMIP/tas/anomalies/ACCESS-ESM1-5/ACCESS-ESM1-5_ssp534-over_r13i1p1f2_tas_anomalies.nc
Skipping (already exists): /home/563/sc1326/repos/cdrmip_extremes/data/processed/ScenarioMIP/tas/anomalies/ACCESS-ESM1-5/ACCESS-ESM1-5_ssp534-over_r14i1p1f2_tas_anomalies.nc
Skipping (already exists): /home/563/sc1326/repos/cdrmip_extremes/data/processed/ScenarioMIP/tas/anomalies/ACCESS-ESM1-5/ACCESS-ES

In [19]:
processing.save_processed(gsat_dt,proc_dir,'ScenarioMIP','anomalies','gsat')

Skipping (already exists): /home/563/sc1326/repos/cdrmip_extremes/data/processed/ScenarioMIP/gsat/anomalies/ACCESS-ESM1-5/ACCESS-ESM1-5_ssp534-over_r10i1p1f2_gsat_anomalies.nc
Skipping (already exists): /home/563/sc1326/repos/cdrmip_extremes/data/processed/ScenarioMIP/gsat/anomalies/ACCESS-ESM1-5/ACCESS-ESM1-5_ssp534-over_r11i1p1f2_gsat_anomalies.nc
Skipping (already exists): /home/563/sc1326/repos/cdrmip_extremes/data/processed/ScenarioMIP/gsat/anomalies/ACCESS-ESM1-5/ACCESS-ESM1-5_ssp534-over_r12i1p1f2_gsat_anomalies.nc
Skipping (already exists): /home/563/sc1326/repos/cdrmip_extremes/data/processed/ScenarioMIP/gsat/anomalies/ACCESS-ESM1-5/ACCESS-ESM1-5_ssp534-over_r13i1p1f2_gsat_anomalies.nc
Skipping (already exists): /home/563/sc1326/repos/cdrmip_extremes/data/processed/ScenarioMIP/gsat/anomalies/ACCESS-ESM1-5/ACCESS-ESM1-5_ssp534-over_r14i1p1f2_gsat_anomalies.nc
Skipping (already exists): /home/563/sc1326/repos/cdrmip_extremes/data/processed/ScenarioMIP/gsat/anomalies/ACCESS-ESM1-